In [1]:
!pip install pandas sqlalchemy requests

In [2]:
import requests

url = "https://github.com/lerocha/chinook-database/raw/master/ChinookDatabase/DataSources/Chinook_Sqlite.sqlite"

response = requests.get(url)

with open("chinook.sqlite", "wb") as f:
    f.write(response.content)

print("Download complete")

Download complete


In [3]:
from sqlalchemy import create_engine

engine = create_engine("sqlite:///chinook.sqlite")

In [4]:
import pandas as pd

pd.read_sql("SELECT * FROM Artist LIMIT 5;", engine)


,ArtistId,Name
0,1,AC/DC
1,2,Accept
2,3,Aerosmith
3,4,Alanis Morissette
4,5,Alice In Chains


In [5]:
query = """
SELECT
    c.FirstName || ' ' || c.LastName AS CustomerName,
    i.InvoiceId,
    t.Name AS TrackName,
    ar.Name AS ArtistName,
    il.UnitPrice,
    il.Quantity
FROM Customer c
JOIN Invoice i ON c.CustomerId = i.CustomerId
JOIN InvoiceLine il ON i.InvoiceId = il.InvoiceId
JOIN Track t ON il.TrackId = t.TrackId
JOIN Album al ON t.AlbumId = al.AlbumId
JOIN Artist ar ON al.ArtistId = ar.ArtistId;
"""

df = pd.read_sql(query, engine)
df.head()

,CustomerName,InvoiceId,TrackName,ArtistName,UnitPrice,Quantity
0,Leonie Köhler,1,Balls to the Wall,Accept,0.99,1
1,Leonie Köhler,1,Restless and Wild,Accept,0.99,1
2,Bjørn Hansen,2,Put The Finger On You,AC/DC,0.99,1
3,Bjørn Hansen,2,Inject The Venom,AC/DC,0.99,1
4,Bjørn Hansen,2,Evil Walks,AC/DC,0.99,1


In [6]:
df.shape
df.columns
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2240 entries, 0 to 2239
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   CustomerName  2240 non-null   object 
 1   InvoiceId     2240 non-null   int64  
 2   TrackName     2240 non-null   object 
 3   ArtistName    2240 non-null   object 
 4   UnitPrice     2240 non-null   float64
 5   Quantity      2240 non-null   int64  
dtypes: float64(1), int64(2), object(3)
memory usage: 105.1+ KB


In [7]:
df["Total"] = df["UnitPrice"] * df["Quantity"]

revenue_per_customer = df.groupby("CustomerName")["Total"].sum().sort_values(ascending=False)

revenue_per_customer.head()

,Total
CustomerName,
Helena Holý,49.62
Richard Cunningham,47.62
Luis Rojas,46.62
Hugh O'Reilly,45.62
Ladislav Kovács,45.62


In [8]:
top_artists = df.groupby("ArtistName")["Quantity"].sum().sort_values(ascending=False)

top_artists.head()

,Quantity
ArtistName,
Iron Maiden,140
U2,107
Metallica,91
Led Zeppelin,87
Os Paralamas Do Sucesso,45
